# SpinStoq: Fast Correlated Noise Generator

Generate spin-qubit noise with explicit timestamps and reproducible seeding.

In [ ]:
import numpy as np
from src.noise import generate, calibrate, welch
import matplotlib.pyplot as plt

## Example 1: Generate 1/f noise by name

In [ ]:
# Generate 100 trajectories, 1e4 Hz sample rate, 2^16 points
res = generate(
    "1/f",
    n_traj=100,
    fs=1e4,               # sample rate [Hz]
    n_points=2**16,
    alpha=1.0,            # 1/f^alpha
    S0=1e-12,             # PSD amplitude [units^2/Hz]
    f0=1.0,               # reference frequency [Hz]
    seed=42,
)

print(f"Trajectories shape: {res.traj.shape}")
print(f"Time grid shape: {res.t.shape}")
print(f"Seed: {res.seed}")
print(f"Spec: {res.spec}")

## Example 2: Inspect timestamps and first trajectory

In [ ]:
# Time grid in seconds
print(f"Time range: {res.t[0]:.6f} to {res.t[-1]:.6f} seconds")
print(f"Time step: {res.t[1] - res.t[0]:.9f} seconds")
print(f"Sample rate: {res.fs} Hz")

# First trajectory
traj_0 = res.traj[0, :]
print(f"\nTrajectory[0] stats: mean={traj_0.mean():.3e}, std={traj_0.std():.3e}")

## Example 3: Plot trajectory and power spectrum

In [ ]:
# Compute PSD via Welch
f, S = welch(res.traj, fs=res.fs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Time domain: first 1000 samples
ax1.plot(res.t[:1000], res.traj[0, :1000], linewidth=0.8)
ax1.set_xlabel('Time [s]')
ax1.set_ylabel('Noise')
ax1.set_title('Sample trajectory (first 1000 samples)')
ax1.grid(alpha=0.3)

# Frequency domain
ax2.loglog(f[1:], S[1:], label='measured', linewidth=1, alpha=0.8)
ax2.loglog(f[1:], 1.0/f[1:], 'k--', alpha=0.5, label='target 1/f')
ax2.set_xlabel('Frequency [Hz]')
ax2.set_ylabel('PSD [units²/Hz]')
ax2.set_title('Power spectrum (all 100 trajectories averaged)')
ax2.legend()
ax2.grid(alpha=0.3, which='both')

fig.tight_layout()
plt.show()

## Example 4: Generate OU-sum (physically-motivated 1/f)

In [ ]:
# OU-sum: sum of Lorentzians (better for physics simulations)
res_ou = generate(
    "ou_sum",
    n_traj=50,
    fs=1e4,
    n_points=2**14,
    alpha=1.0,
    S0=1e-12,
    f0=1.0,
    n_components_per_decade=8,
    seed=42,
    backend="numpy",  # or "numba" for speed
)

print(f"OU-sum trajectories: {res_ou.traj.shape}")
print(f"Process spec: {res_ou.spec['process']}")

## Example 5: From measured trace (model-free surrogate)

In [ ]:
# Create a synthetic measured trace
trace_measured = np.random.randn(1, 2**14)  # 1 channel, 16384 samples

# Calibrate: estimate statistics and create surrogate generator
gen = calibrate(trace_measured, fs=1e4, method="circulant")

# Generate 200 surrogate trajectories matching the measured PSD
res_surrogate = gen.sample(n_traj=200, n_points=2**14, seed=42)

print(f"Surrogate trajectories: {res_surrogate.traj.shape}")
print(f"Timestamps included: {res_surrogate.t.shape}")

## Example 6: List available named processes

In [ ]:
from src.noise import list_processes

processes = list_processes()
print("Available noise processes:")
for name in processes:
    print(f"  - {name}")

## Example 7: Save and load results

In [ ]:
import tempfile
import os

# Save to .npz file (includes metadata)
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "noise_result.npz")
    res.save(path)
    print(f"Saved to {path}")
    
    # Load it back
    from src.noise import NoiseResult
    res_loaded = NoiseResult.load(path)
    
    print(f"Loaded shape: {res_loaded.traj.shape}")
    print(f"Metadata preserved: seed={res_loaded.seed}, fs={res_loaded.fs}")